# 1.0 Converting TEMPO files to parquet files for all hours

This notebook converts TEMPO L3 V04 NetCDF files into Parquet for use with pandas/dask, the output is one Parquet dataset folder per month. Just need to change the folder path to run for another year.

Steps:
1) Lazily opens the product, support_data, geolocation, and root groups of each file
2) Builds a dataset with NO2 tropospheric column, NO2 uncertainty, quality flag, effective cloud fraction, AMF cloud fraction, snow/ice fraction, solar zenith angle, latitude, longitude, and time
3) Masks to the study region (−104 to −80°E, 36 to 49°N) and keeps pixels with quality flag ≠ 1 or 2 and NO2 > 0 
4) Writes one Parquet file per input file to a temporary folder
5) Combines these into one Parquet dataset folder per month and deletes the temporary folder

In [1]:
import os
import shutil
import re
from pathlib import Path
from datetime import datetime
import pandas as pd
import xarray as xr
from tqdm import tqdm
import dask.dataframe as dd
import pyarrow.parquet as pq
import pyarrow as pa

In [2]:
BASE_DIR = Path("/global/scratch/users/rubypercival/TEMPO24/")
OUTPUT_BASE = Path("/global/scratch/users/rubypercival/TEMPO24/parquet_V04")
nc_files = sorted(BASE_DIR.glob("*.nc"))

lon_min, lon_max = -104, -80
lat_min, lat_max = 36, 49

YEAR = 2024
MONTHS = [3, 4, 5, 6, 7]

print("Input:", BASE_DIR)
print("Output:", OUTPUT_BASE)
print("Input exists:", BASE_DIR.exists())

Input: /global/scratch/users/rubypercival/TEMPO24
Output: /global/scratch/users/rubypercival/TEMPO24/parquet_V04
Input exists: True


In [3]:
filename_date_pattern = re.compile(r"V04_(\d{8})T")

def get_file_date(file_path):
    match = filename_date_pattern.search(Path(file_path).name)
    if match is None:
        return None
    return datetime.strptime(match.group(1), "%Y%m%d")

selected_files = [
    f for f in nc_files
    if get_file_date(f) is not None
    and get_file_date(f).year == YEAR
    and get_file_date(f).month in MONTHS]

print(f"Found {len(selected_files)} files for {YEAR}, March–July.")

for month in MONTHS:
    n = sum(get_file_date(f).month == month for f in selected_files)
    print(f"{YEAR}-{month:02d}: {n} files")

Found 2444 files for 2024, March–July.
2024-03: 434 files
2024-04: 454 files
2024-05: 529 files
2024-06: 509 files
2024-07: 518 files


## Conversion function

In [7]:
def process_no2_files_to_combined_parquet(
    folder_path,
    output_parquet,
    lon_min,
    lon_max,
    lat_min,
    lat_max,
    files=None):
    folder_path = Path(folder_path)
    output_parquet = Path(output_parquet)

    temp_parquet_dir = output_parquet.parent / f"_temp_parquets_{output_parquet.name}"
    if temp_parquet_dir.exists():
        shutil.rmtree(temp_parquet_dir)
    temp_parquet_dir.mkdir(parents=True, exist_ok=True)

    if files is None:
        files = sorted(folder_path.glob("*.nc"))

    files = [Path(f) for f in files]

    print(f"Processing {len(files)} NetCDF files")
    successful = 0
    failed = 0

    for file_path in tqdm(files, desc="Processing Files", unit="file"):
        try:
           with xr.open_dataset(file_path, group='product', decode_times=False, engine="netcdf4", chunks={}) as product_ds, \
                 xr.open_dataset(file_path, group='support_data', decode_times=False, engine="netcdf4", chunks={}) as support_ds, \
                 xr.open_dataset(file_path, group='geolocation', decode_times=False, engine='netcdf4', chunks={}) as geolocation_ds, \
                 xr.open_dataset(file_path, group=None, decode_times=False, engine="netcdf4", chunks={}) as root_ds:

                no2 = product_ds["vertical_column_troposphere"]
                uncertainty = product_ds["vertical_column_troposphere_uncertainty"]
                flag = product_ds["main_data_quality_flag"]

                eff_cloud = support_ds["eff_cloud_fraction"]
                amf_cloud = support_ds["amf_cloud_fraction"]
                snow_ice = support_ds["snow_ice_fraction"]

                solar_zenith = geolocation_ds["solar_zenith_angle"]

                lats = root_ds["latitude"]
                lons = root_ds["longitude"]
                time_raw = root_ds["time"]

                time = xr.decode_cf(
                    xr.Dataset({"time": time_raw}))["time"]

                ds = xr.Dataset({
                    "no2": no2,
                    "no2_uncertainty": uncertainty,
                    "flag": flag,
                    "eff_cloud_fraction": eff_cloud,
                    "amf_cloud_fraction": amf_cloud,
                    "snow_ice_fraction": snow_ice,
                    "solar_zenith_angle": solar_zenith,
                    "latitude": lats,
                    "longitude": lons,
                    "time": time})

                ds = ds.unify_chunks()

                if "latitude" in ds.dims and "longitude" in ds.dims:
                    ds = ds.chunk({"latitude": 512, "longitude": 512})

                mask = (
                    (ds["flag"] != 1) &
                    (ds["flag"] != 2) &
                    (ds["no2"] > 0) &
                    (ds["longitude"] >= lon_min) &
                    (ds["longitude"] <= lon_max) &
                    (ds["latitude"] >= lat_min) &
                    (ds["latitude"] <= lat_max))

                mask = mask.compute()
                filtered = ds.where(mask, drop=True)
                pdf = filtered.to_dataframe().reset_index()

                pdf["time"] = pd.to_datetime(pdf["time"], errors="coerce")
                pdf = pdf[pdf["time"].notnull()]

                pdf["date"] = pdf["time"].dt.date.astype(str)
                pdf["time"] = pdf["time"].dt.time.astype(str)

                if "flag" in pdf.columns:
                    pdf = pdf.drop(columns=["flag"])

                output_file = temp_parquet_dir / f"{file_path.stem}.parquet"

                pdf.to_parquet(output_file,index=False,engine="pyarrow")
                successful += 1

        except Exception as e:
            failed += 1
            print(f"\nERROR processing {file_path.name}: {e}")

    print("\nFinished individual file conversion.")
    print(f"Successful: {successful}")
    print(f"Failed: {failed}")
    print("\nMaking combined Parquet dataset...")

    output_parquet.parent.mkdir(parents=True, exist_ok=True)
    if output_parquet.exists():
        shutil.rmtree(output_parquet)

    final_ddf = dd.read_parquet(str(temp_parquet_dir),engine="pyarrow",index=None)
    final_ddf.to_parquet(str(output_parquet),engine="pyarrow",write_index=False)

    print(f"All NO2 data saved to: {output_parquet}")
    shutil.rmtree(temp_parquet_dir)

## Process March–July 2024

In [14]:
for month in MONTHS:

    month_files = [
        f for f in selected_files
        if get_file_date(f).month == month]

    month_output = OUTPUT_BASE / f"all_tempo_hours_no2_L3_V04_{YEAR}_{month:02d}_folder"

    print("\n" + "=" * 70)
    print(f"PROCESSING {YEAR}-{month:02d}")
    print(f"Number of files: {len(month_files)}")
    print(f"Output: {month_output}")
    print("=" * 70)

    process_no2_files_to_combined_parquet(
        BASE_DIR,
        month_output,
        lon_min,
        lon_max,
        lat_min,
        lat_max,
        files=month_files)


PROCESSING 2024-03
Number of files: 434
Output: /global/scratch/users/rubypercival/TEMPO24/parquet_V04/all_tempo_hours_no2_L3_V04_2024_03_folder
Processing 434 NetCDF files


Processing Files: 100%|██████████| 434/434 [30:28<00:00,  4.21s/file]



Finished individual file conversion.
Successful: 434
Failed: 0

Making combined Parquet dataset...
All NO2 data saved to: /global/scratch/users/rubypercival/TEMPO24/parquet_V04/all_tempo_hours_no2_L3_V04_2024_03_folder

PROCESSING 2024-04
Number of files: 454
Output: /global/scratch/users/rubypercival/TEMPO24/parquet_V04/all_tempo_hours_no2_L3_V04_2024_04_folder
Processing 454 NetCDF files


Processing Files: 100%|██████████| 454/454 [32:32<00:00,  4.30s/file]



Finished individual file conversion.
Successful: 454
Failed: 0

Making combined Parquet dataset...
All NO2 data saved to: /global/scratch/users/rubypercival/TEMPO24/parquet_V04/all_tempo_hours_no2_L3_V04_2024_04_folder

PROCESSING 2024-05
Number of files: 529
Output: /global/scratch/users/rubypercival/TEMPO24/parquet_V04/all_tempo_hours_no2_L3_V04_2024_05_folder
Processing 529 NetCDF files


Processing Files: 100%|██████████| 529/529 [37:42<00:00,  4.28s/file]



Finished individual file conversion.
Successful: 529
Failed: 0

Making combined Parquet dataset...
All NO2 data saved to: /global/scratch/users/rubypercival/TEMPO24/parquet_V04/all_tempo_hours_no2_L3_V04_2024_05_folder

PROCESSING 2024-06
Number of files: 509
Output: /global/scratch/users/rubypercival/TEMPO24/parquet_V04/all_tempo_hours_no2_L3_V04_2024_06_folder
Processing 509 NetCDF files


Processing Files: 100%|██████████| 509/509 [36:45<00:00,  4.33s/file]



Finished individual file conversion.
Successful: 509
Failed: 0

Making combined Parquet dataset...
All NO2 data saved to: /global/scratch/users/rubypercival/TEMPO24/parquet_V04/all_tempo_hours_no2_L3_V04_2024_06_folder

PROCESSING 2024-07
Number of files: 518
Output: /global/scratch/users/rubypercival/TEMPO24/parquet_V04/all_tempo_hours_no2_L3_V04_2024_07_folder
Processing 518 NetCDF files


Processing Files: 100%|██████████| 518/518 [38:12<00:00,  4.42s/file]



Finished individual file conversion.
Successful: 518
Failed: 0

Making combined Parquet dataset...
All NO2 data saved to: /global/scratch/users/rubypercival/TEMPO24/parquet_V04/all_tempo_hours_no2_L3_V04_2024_07_folder
